<div align="center">
  <h2>Міністерство освіти і науки України</h2>
  <h2>Національний технічний університет України</h2>
  <h2>«Київський політехнічний інститут ім. Ігоря Сікорського»</h2>
  <h2>Факультет інформатики та обчислювальної техніки</h2>
  <h2>Кафедра обчислювальної техніки</h2>
  <br>
</div>

<div align="right">
    <br>
    <br>
<center>
<h2>Комп'ютерний практикум №14</h2>
    <h2>з дисципліни</h2>
    <h2>«Штучний інтелект в задачах обробки зображень»</h2>
    <h2>на тему:</h2>
    <h2>«Розпізнавання номерних знаків за допомогою Tesseract»</h2>
</center>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
Виконали: <br>
Студенти ІІІ курсу ФІОТ <br>
групи ІО-34 <br>
Токарюк С. Б., Рибачок М. В.<br>
</div>

<center>
Київ - 2026
</center>


## 1. Мета роботи
Навчитися отримувати текст номерного знаку з зображень використовуючи можливості OpenCV та Tesseract.

## 2. Програма роботи

1) **Встановлення та імпортування необхідних бібліотек** — налаштування середовища для OpenCV, PyTesseract, imutils та skimage.
2) **Створення методу для знаходження контурів** — реалізація морфологічних операцій (blackhat, закриття) та градієнта Шарра для локалізації потенційних номерних знаків.
3) **Знаходження найбільш ймовірного контуру** — фільтрація знайдених контурів за співвідношенням сторін та виділення регіону інтересу (ROI).
4) **Налаштування для PyTesseract** — визначення методу сегментації сторінки (PSM) та створення білого списку символів (whitelist) для покращення якості OCR.
5) **Конвертація зображення та розпізнавання** — об'єднання всіх кроків у єдиний конвеєр, обробка вхідного зображення, розпізнавання тексту та виведення результату.

---
# 3. ТЕОРЕТИЧНІ ВІДОМОСТІ

## Що означає розпізнавання тексту?
Метод вилучення тексту із зображень називається оптичним розпізнаванням символів (OCR, Optical Character Recognition) або просто розпізнаванням тексту. Ви можете розглядати виявлення тексту як спеціалізовану форму виявлення об’єктів. Під час виявлення тексту мета полягає в автоматичному обчисленні обмежувальних рамок для кожної області тексту на зображенні. Коли текст виявлено на зображенні, ми можемо декодувати його за допомогою програмного забезпечення OCR.

## Автоматичне розпізнавання номерних знаків (ANPR/ALPR)
Це процес, що складається з кількох етапів:
1. Виявлення та локалізація номерного знаку на вхідному зображенні/кадрі;
2. Витягнення символів з номерного знаку;
3. Застосування певної форми OCR для розпізнавання витягнутих символів.

Автоматичні системи розпізнавання номерних знаків бувають різних форм і розмірів:
- Системи, що працюють в контрольованих умовах освітлення з передбачуваними типами номерних знаків, можуть використовувати базові методи обробки зображень.
- Більш досконалі системи ANPR використовують спеціальні детектори об'єктів (HOG + Linear SVM, Faster R-CNN, SSDs і YOLO).
- Найсучасніше ПЗ використовує рекурентні нейронні мережі (RNN) та LSTM для кращого розпізнавання.
- Спеціалізовані архітектури нейронних мереж виконують попередню обробку та очищення зображень перед розпізнаванням для підвищення точності.

---
# 4. Виконання роботи

## Крок 1. Встановлення та імпортування необхідних бібліотек

In [7]:
import cv2
import imutils
import numpy as np
from skimage.segmentation import clear_border
from imutils import paths
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

## Крок 2. Створення методу для знаходження контурів
Метод `locate_plate_candidates` приймає зображення у відтінках сірого. За допомогою морфологічної операції `blackhat` виділяються темні символи на світлому фоні. Далі градієнт Шарра виявляє краї, а розмиття Гаусса та бінарний поріг Оцу очищують зображення для надійного пошуку контурів.

In [8]:
def locate_plate_candidates(gray):
    rect_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (14, 5))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, rect_kern)
    
    grad_x = cv2.Sobel(blackhat, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=-1)
    grad_x = np.absolute(grad_x)
    (minVal, maxVal) = (np.min(grad_x), np.max(grad_x))
    grad_x = 255 * ((grad_x - minVal) / (maxVal - minVal))
    grad_x = grad_x.astype("uint8")
    
    grad_x = cv2.GaussianBlur(grad_x, (5, 5), 0)
    grad_x = cv2.morphologyEx(grad_x, cv2.MORPH_CLOSE, rect_kern)
    thresh = cv2.threshold(grad_x, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)
    
    square_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 1))
    light = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, square_kern)
    light = cv2.threshold(light, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    thresh = cv2.bitwise_and(thresh, thresh, mask=light)
    thresh = cv2.dilate(thresh, None, iterations=2)
    thresh = cv2.erode(thresh, None, iterations=1)
    
    contours = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = imutils.grab_contours(contours)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
    
    return contours

## Крок 3. Знаходження найбільш ймовірного контуру
Цикл перебору кандидатів має на меті ізолювати контур, що містить номерний знак. Якщо співвідношення сторін (`min_ar` та `max_ar`) обмежувальної рамки контуру підходить, витягується область інтересу (ROI) та очищається від шуму на межах за допомогою `clear_border`.

In [9]:
def locate_plate(gray, candidates, min_ar=4, max_ar=5):
    contour = None
    interest_reg = None
    
    for candidate in candidates:
        (x, y, w, h) = cv2.boundingRect(candidate)
        ar = w / float(h)
        
        if min_ar <= ar <= max_ar:
            contour = candidate
            license_plate = gray[y:y + h, x:x + w]
            interest_reg = cv2.threshold(license_plate, 0, 255,
                                         cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
            interest_reg = clear_border(interest_reg)
            break
            
    return interest_reg, contour

## Крок 4. Налаштування для PyTesseract
Метод сегментації сторінки (PSM = 7) розглядає зображення як один текстовий рядок. Також налаштовується білий список (whitelist), щоб відкинути небажані символи, і створюється функція для очищення non-ASCII символів.

In [10]:
def build_tesseract_options(psm=7):
    alphanumeric = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    options = "-c tessedit_char_whitelist={}".format(alphanumeric)
    options += " --psm {}".format(psm)
    return options

def cleanup_text(text):
    return "".join([c if ord(c) < 128 else "" for c in text]).strip()

## Крок 5. Конвертація зображення та виконання конвеєра
Завантажуємо зображення з папки `group`, конвертуємо у відтінки сірого, визначаємо набір контурів-кандидатів, локалізуємо номер і передаємо його в Tesseract для зчитування тексту.

In [22]:
imagePaths = sorted(list(paths.list_images("group")))

for imagePath in imagePaths:
    image = cv2.imread(imagePath)
    image = imutils.resize(image, width=600)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    candidates = locate_plate_candidates(gray)
    interest_reg, contour = locate_plate(gray, candidates, min_ar=1, max_ar=6)
    
    if interest_reg is not None:
        options = build_tesseract_options(psm=7)
        text = pytesseract.image_to_string(interest_reg, config=options)
        
        if text is not None and contour is not None:
            box = cv2.boxPoints(cv2.minAreaRect(contour)).astype("int")
            cv2.drawContours(image, [box], -1, (0, 255, 0), 2)
            
            (x, y, w, h) = cv2.boundingRect(contour)
            cv2.putText(image, cleanup_text(text), (x, y - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
            
    cv2.imshow("Output ANPR", image)
    cv2.waitKey(0)

cv2.destroyAllWindows()

## Висновок:
Під час виконання лабораторної роботи було створено базову систему автоматичного розпізнавання номерних знаків (ANPR) за допомогою Python, OpenCV та PyTesseract. Ми на практиці застосували методи попередньої обробки зображень (морфологічні операції, розмиття Гаусса, алгоритм Шарра) та пошуку контурів для точної локалізації регіону інтересу (ROI) — номерного знаку автомобіля.

Крім того, було досліджено можливості рушія Tesseract OCR для витягнення тексту зі знайденого регіону. Налаштування параметрів Tesseract (наприклад, методу сегментації сторінки PSM та білого списку символів) разом із додатковим очищенням результатів суттєво підвищили точність зчитування, довівши ефективність комбінування комп'ютерного зору та методів OCR.